# Hotel Booking Cancellation & Overbooking Decision
## 06 — Baseline Model Training

Train Logistic Regression, KNN, Decision Tree, Random Forest and HistGradientBoosting, then compare classification performance.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE=42

## 2. Load Cleaned Data

In [ ]:
DATA_PATH='hotel_bookings.csv'
df=pd.read_csv(DATA_PATH)
print('Shape:',df.shape)
df.head()

## 3. Recreate Feature Engineering

In [ ]:
df_model=df.copy()
df_model['total_nights']=df_model['stays_in_weekend_nights']+df_model['stays_in_week_nights']
df_model['total_guests']=df_model['adults']+df_model['children']+df_model['babies']
df_model['total_previous_bookings']=df_model['previous_cancellations']+df_model['previous_bookings_not_canceled']
df_model['has_special_request']=(df_model['total_of_special_requests']>0).astype(int)
df_model['room_type_changed']=(df_model['reserved_room_type']!=df_model['assigned_room_type']).astype(int)
df_model=df_model.drop(columns=['stays_in_weekend_nights','stays_in_week_nights','adults','children','babies'])
print(df_model.shape)

## 4. Define X and y + Stratified Split

In [ ]:
X=df_model.drop(columns=['is_canceled'])
y=df_model['is_canceled']

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)

print('Train:',X_train.shape)
print('Test:',X_test.shape)
print('Train cancellation %:',round(y_train.mean()*100,2))
print('Test cancellation %:',round(y_test.mean()*100,2))

## 5. Preprocessing

In [ ]:
num=X_train.select_dtypes(include=np.number).columns.tolist()
cat=X_train.select_dtypes(include=['object','category']).columns.tolist()

num_pipe=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
cat_pipe=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder(handle_unknown='ignore'))
])
preprocessor=ColumnTransformer([
    ('num',num_pipe,num),
    ('cat',cat_pipe,cat)
])

boost_preprocessor=ColumnTransformer([
    ('num',SimpleImputer(strategy='median'),num),
    ('cat',Pipeline([
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))
    ]),cat)
])

print('Numerical:',len(num),'Categorical:',len(cat))

## 6. Define Baseline Models

In [ ]:
models={
 'Logistic Regression':Pipeline([
     ('preprocessor',preprocessor),
     ('model',LogisticRegression(max_iter=1000,random_state=RANDOM_STATE))
 ]),
 'KNN':Pipeline([
     ('preprocessor',preprocessor),
     ('model',KNeighborsClassifier(n_neighbors=5,n_jobs=-1))
 ]),
 'Decision Tree':Pipeline([
     ('preprocessor',preprocessor),
     ('model',DecisionTreeClassifier(random_state=RANDOM_STATE))
 ]),
 'Random Forest':Pipeline([
     ('preprocessor',preprocessor),
     ('model',RandomForestClassifier(n_estimators=100,random_state=RANDOM_STATE,n_jobs=-1))
 ]),
 'HistGradientBoosting':Pipeline([
     ('preprocessor',boost_preprocessor),
     ('model',HistGradientBoostingClassifier(random_state=RANDOM_STATE))
 ])
}
print(list(models.keys()))

## 7. Train and Evaluate

PR-AUC is reported using Average Precision. These are baseline models; hyperparameter tuning comes later.

In [ ]:
results=[]
predictions={}
probabilities={}
fitted_models={}

for name,model in models.items():
    print('Training',name,'...')
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    prob=model.predict_proba(X_test)[:,1]

    fitted_models[name]=model
    predictions[name]=pred
    probabilities[name]=prob

    results.append({
        'Model':name,
        'Accuracy':accuracy_score(y_test,pred),
        'Precision':precision_score(y_test,pred,zero_division=0),
        'Recall':recall_score(y_test,pred,zero_division=0),
        'F1':f1_score(y_test,pred,zero_division=0),
        'ROC_AUC':roc_auc_score(y_test,prob),
        'PR_AUC':average_precision_score(y_test,prob)
    })

results_df=pd.DataFrame(results)
results_df

## 8. Comparison Table

In [ ]:
metrics=['Accuracy','Precision','Recall','F1','ROC_AUC','PR_AUC']
comparison=results_df.copy()
comparison[metrics]=comparison[metrics].round(4)
comparison.sort_values('ROC_AUC',ascending=False)

## 9. Visual Comparison

In [ ]:
comparison.set_index('Model')[metrics].plot(kind='bar',figsize=(12,6))
plt.title('Baseline Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0,1)
plt.xticks(rotation=25,ha='right')
plt.tight_layout()
plt.show()

## 10. Confusion Matrices

In [ ]:
for name in models:
    cm=confusion_matrix(y_test,predictions[name])
    ConfusionMatrixDisplay(cm,display_labels=['Not Cancelled','Cancelled']).plot(values_format='d')
    plt.title('Confusion Matrix - '+name)
    plt.tight_layout()
    plt.show()

## 11. Strongest Baseline by ROC-AUC

In [ ]:
best=results_df.loc[results_df['ROC_AUC'].idxmax()]
print('Model:',best['Model'])
print('ROC-AUC:',round(best['ROC_AUC'],4))
print('PR-AUC:',round(best['PR_AUC'],4))
print('F1:',round(best['F1'],4))

## Error Meaning

- **TP:** predicted cancellation and booking cancelled.
- **TN:** predicted no cancellation and booking did not cancel.
- **FP:** predicted cancellation but guest did not cancel.
- **FN:** predicted no cancellation but booking cancelled.

For overbooking, a false positive can be risky because the hotel may expect a cancellation that does not occur.